# 24 - Guardrails & Policy Enforcement

## Scenario: Real-time Output Scrubbing

Even with a perfect system prompt, an LLM might accidentally leak PII (Personally Identifiable Information) that it fetched from a database tool.
We cannot rely on the LLM to police itself perfectly. We must implement **Deterministic Guardrails**.

In this notebook, we build a pipeline that intercepts the Agent's final output and scrubs any Social Security Numbers (SSNs) or Credit Cards *before* returning it to the user.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Defining the Deterministic Guardrail

We use a simple regex-based scrubber (simulating enterprise tools like Presidio or NeMo Guardrails).

In [2]:
import re

def scrub_pii(text: str) -> str:
    """Scrubs SSNs and Credit Card patterns from text."""
    # Simple regex for SSN (XXX-XX-XXXX)
    ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"
    # Simple regex for 16-digit CC
    cc_pattern = r"\b\d{4}-\d{4}-\d{4}-\d{4}\b"
    
    clean_text = re.sub(ssn_pattern, "[REDACTED SSN]", text)
    clean_text = re.sub(cc_pattern, "[REDACTED CC]", clean_text)
    
    if clean_text != text:
        print("🛡️ [Guardrail] PII Leak detected and scrubbed!")
        
    return clean_text


## 2. The Guardrailed Agent Pipeline

In [3]:
def support_agent_pipeline(query: str):
    print(f"📩 Query: {query}")
    
    # 1. Agent Generates Answer (Mocked for demonstration)
    print("🧠 [Agent] Generating response...")
    raw_response = "I have updated your account. For your records, the SSN on file is 123-45-6789 and the card is 4111-2222-3333-4444."
    
    # 2. Guardrail Intercepts
    safe_response = scrub_pii(raw_response)
    
    # 3. Final Output
    print(f"📤 Final Output to User: {safe_response}")

support_agent_pipeline("Can you confirm my account details?")


📩 Query: Can you confirm my account details?
🧠 [Agent] Generating response...
🛡️ [Guardrail] PII Leak detected and scrubbed!
📤 Final Output to User: I have updated your account. For your records, the SSN on file is [REDACTED SSN] and the card is [REDACTED CC].


## Checkpoint

**1. Why use deterministic regex/Presidio for PII scrubbing instead of just asking the LLM not to output PII?**
- A) Deterministic code is faster.
- B) LLMs are probabilistic and prone to jailbreaks or hallucinations. A deterministic guardrail guarantees that known PII patterns will *never* reach the user, regardless of what the LLM decides.
- C) Regex understands context better than LLMs.
- D) It looks cooler.
